# Controlled Markov Chains

**Reinforcement Learning Course** — Notebook 02

We study how a policy $\pi$ shapes the induced chain $P^\pi$,
its stationary distribution $\mu_\infty$, the occupation measure $\rho$,
and the discounted occupation measure $d^\pi_\gamma$.

In [ ]:
import os
if os.getenv("COLAB_RELEASE_TAG"):
    !git clone --depth 1 https://github.com/Paul-antoineLeTolguenec/rl-course.git /content/rl-course
    os.chdir("/content/rl-course/notebooks")
%pip install -q numpy plotly nbformat

In [2]:
import sys; sys.path.insert(0, "..")
from practical_case import markov_env as env

## Part 1 — The grid environment

A 20×20 grid world with wall cells (violet). The agent moves in 8 directions.

In [3]:
env.show_grid(title="Grid world").show()

## Part 2 — Policy shapes the dynamics

Three policies → three different ^\pi$ → three different $\mu_t^\pi$.

In [4]:
policies = {
    "Spiral CCW": env.spiral_ccw(),
    "Spiral CW":  env.spiral_cw(),
    "Rightward":  env.rightward(),
}

for name, pi in policies.items():
    P = env.induced_chain(pi,alpha=0.4)
    env.show_grid(P=P, title=name).show()

In [5]:
# Evolution of mu_t under the CCW spiral policy
P_ccw = env.induced_chain(env.spiral_ccw())
env.animate(P_ccw).show()

## Part 3 — Stationary distribution & mixing time

In practice we don't know $\mu_\infty$ in advance — but we can monitor convergence via the **incremental criterion** $\|\mu_{t+1} - \mu_t\|_1 \to 0$.

Both criteria converge simultaneously (left: vs $\mu_\infty$, right: incremental). This raises the question: **how do we compute $\mu_\infty$?** → power iteration on $P^\pi$ (see cell below).

In [6]:
mu_inf = env.stationary(P_ccw)
env.show_grid(mu=mu_inf, title="μ_∞ — Spiral CCW").show()

In [7]:
env.mixing_plot(policies).show()

## Part 4 — Occupation measure

$$\rho(s) = \lim_{T \to \infty} \frac{1}{T} \sum_{t=0}^{T-1} \mu_t^\pi(s)$$

The running average $\rho_T$ converges to $\mu_\infty$ — visualised frame by frame.

In [8]:
env.rho_animation(P_ccw,n_frames=500).show()

## Part 5 — Discounted occupation measure

$$d^\pi_\gamma(s) = (1-\gamma)\sum_{t=0}^\infty \gamma^t\, \mu_t^\pi(s)$$

As $\gamma \to 1$, $d^\pi_\gamma \to \mu_\infty = \rho$. Small $\gamma$ weights early states more.

In [9]:
env.occupation_plot(P_ccw).show()

## Part 6 — (Optional) Your own policy

Return a valid $\pi$ array of shape `(N, A_DIM)` with rows summing to 1.
Wall states must be uniform: `pi[s] = 1/A_DIM` for `s in WALL_STATES`.

In [10]:
import numpy as np

def my_policy():
    pi = np.zeros((env.N, env.A_DIM))
    # ── your code here ────────────────────────────────────────────────

    # ─────────────────────────────────────────────────────────────────
    for s in env.WALL_STATES: pi[s] = 1.0 / env.A_DIM
    assert np.allclose(pi.sum(axis=1), 1.0), "rows must sum to 1"
    return pi

# Uncomment once implemented:
# P_custom = env.induced_chain(my_policy())
# env.show_grid(P=P_custom, title="My policy").show()
# env.animate(P_custom).show()
# env.rho_animation(P_custom).show()